In [ ]:
import os
import sys

# 1. Thiết lập các biến môi trường cốt lõi
spark_home = os.path.expanduser("~/spark")
os.environ["SPARK_HOME"] = spark_home
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# 2. Ép PySpark dùng đúng Python 3.10 của môi trường ảo
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 3. Nạp thủ công các file Jar của Spark vào hệ thống Python
# Đây là bước quan trọng nhất để sửa lỗi 'JavaPackage'
import glob
spark_python_path = os.path.join(spark_home, 'python')
py4j_zip = glob.glob(os.path.join(spark_home, 'python/lib', 'py4j-*-src.zip'))[0]

sys.path.insert(0, spark_python_path)
sys.path.insert(0, py4j_zip)

# 4. Khởi tạo với cấu hình "Cưỡng chế"
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

try:
    # Xóa các session cũ đang bị treo (nếu có)
    if 'spark' in locals():
        spark.stop()
    
    conf = SparkConf()
    conf.setMaster("local[*]")
    conf.setAppName("SentimentAnalysis")
    conf.set("spark.driver.bindAddress", "127.0.0.1")
    
    spark = SparkSession.builder.appName("Streaming from kafka")\
    .config("spark.streaming.stopGracefullyOnShutdown", "true")\
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5")\
    .getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    
    print("🎉 CHIẾN THẮNG! Spark đã khởi động thành công.")
    print(f"Phiên bản Spark: {spark.version}")
    
except Exception as e:
    print(f"❌ Vẫn kẹt ở lỗi: {e}")
    print("\n--- Gợi ý kiểm tra cuối cùng ---")
    print(f"Thư mục Spark hiện tại: {spark_home}")
    print(f"File Py4J đang dùng: {py4j_zip}")

26/06/02 08:52:41 WARN Utils: Your hostname, tuong-van-VMware-Virtual-Platform resolves to a loopback address: 127.0.1.1; using 192.168.198.129 instead (on interface ens33)
26/06/02 08:52:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/tuong-van/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/tuong-van/.ivy2/cache
The jars for the packages stored in: /home/tuong-van/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-55ca64ef-6275-447a-b287-02096239d955;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.5 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.5 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 420ms :: artifacts dl 35

🎉 CHIẾN THẮNG! Spark đã khởi động thành công.
Phiên bản Spark: 3.5.5


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, from_json ,cast,col,round
from pyspark.sql.functions import explode  
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
from pyspark.sql.functions import avg

kafka_df = (spark.read.format("kafka")\
           .option("kafka.bootstrap.servers","10.10.12.61:9093")\
           .option("subscribe","vannt_finance")\
           .option("startingOffsets","earliest")\
           .load() )        

# #3.Doc du lieu
# kafka_df.printSchema()
# kafka_df.show(5)

#4.parse data
values_df = kafka_df.withColumn("value",expr("cast(value as string)"))
# have already change the value to float in the producer, it's just a test
schema_kline = StructType([
    StructField("symbol", StringType(), True),
    StructField("start_time", LongType(), True),
    StructField("open_price", StringType(), True),
    StructField("high_price", StringType(), True),
    StructField("low_price", StringType(), True),
    StructField("close_price", StringType(), True),
    StructField("volume", StringType(), True)
]) 
values_json = values_df.withColumn("parsed",from_json("value", schema_kline)).select("parsed")
values_json.show(5,truncate=False)


#5.flatten
flatten_df = values_json.select("parsed.*").alias("data")
flatten_df = flatten_df.withColumn("start_time", expr("cast(start_time/1000 as timestamp)") )
flatten_df = flatten_df.withColumn("open_price", expr("cast(open_price as double)")) \
                       .withColumn("high_price", expr("cast(high_price as double)")) \
                       .withColumn("low_price", expr("cast(low_price as double)")) \
                       .withColumn("close_price", expr("cast(close_price as double)")) \
                       .withColumn("volume", round(col("volume").cast("double"),3))

#7 
watermark_df = flatten_df.withWatermark("start_time", "1 minute")
watermark_df.show()


+----------------------------------------------------------------------------------------------------+
|parsed                                                                                              |
+----------------------------------------------------------------------------------------------------+
|{BTCUSDT, 1774955400000, 66248.80000000, 66298.32000000, 66243.65000000, 66298.31000000, 6.17866000}|
|{BTCUSDT, 1774955460000, 66301.27000000, 66308.00000000, 66301.27000000, 66308.00000000, 0.13536000}|
|{ETHUSDT, 1774955460000, 2025.65000000, 2025.91000000, 2025.64000000, 2025.91000000, 8.58500000}    |
|{BTCUSDT, 1775186580000, 66717.32000000, 66717.32000000, 66717.31000000, 66717.31000000, 0.17667000}|
|{ETHUSDT, 1775186580000, 2057.01000000, 2057.06000000, 2057.01000000, 2057.06000000, 0.42180000}    |
+----------------------------------------------------------------------------------------------------+
only showing top 5 rows

+-------+-------------------+----------+--------

In [ ]:
from pyspark.sql.functions import window, avg
sma_5m = watermark_df.groupBy("symbol",window("start_time","5 minutes")).agg(avg("close_price").alias("SMA_5m"))
sma_5m.show(truncate= False)

+-------+------------------------------------------+------------------+
|symbol |window                                    |SMA_5m            |
+-------+------------------------------------------+------------------+
|ETHUSDT|{2026-05-08 11:40:00, 2026-05-08 11:45:00}|2276.876153846154 |
|ETHUSDT|{2026-05-13 16:20:00, 2026-05-13 16:25:00}|2319.186129032258 |
|BTCUSDT|{2026-05-20 10:40:00, 2026-05-20 10:45:00}|76747.9183076923  |
|ETHUSDT|{2026-05-20 10:45:00, 2026-05-20 10:50:00}|2111.5512745098035|
|BTCUSDT|{2026-05-20 12:20:00, 2026-05-20 12:25:00}|77145.35132075471 |
|ETHUSDT|{2026-03-31 18:10:00, 2026-03-31 18:15:00}|2025.7800000000002|
|ETHUSDT|{2026-05-20 10:55:00, 2026-05-20 11:00:00}|2111.8327272727274|
|BTCUSDT|{2026-05-08 11:40:00, 2026-05-08 11:45:00}|79557.31883116883 |
|ETHUSDT|{2026-04-24 17:35:00, 2026-04-24 17:40:00}|2315.0103875968994|
|ETHUSDT|{2026-05-20 10:50:00, 2026-05-20 10:55:00}|2111.1980612244897|
|BTCUSDT|{2026-05-20 10:45:00, 2026-05-20 10:50:00}|76744.444761

In [ ]:
from pyspark.sql.functions import lag,col,when,abs,round,avg,window
from pyspark.sql.window import Window
def caculate_RSI(batch_df):
    #RSI
    window_spec = Window.partitionBy("symbol").orderBy("start_time")    
    #Tinh gia chenh lech
    df_step1 = batch_df.withColumn("prev_close_price", lag("close_price",1).over(window_spec))
    df_step1 = df_step1.withColumn("change_price",round(col("close_price")-col("prev_close_price"),3))
    
    #Tach biet tang giam
    df_step2 = df_step1.withColumn("gain_close_price",\
                                   when(col("change_price")>=0,col("change_price")).otherwise(0))
    df_step2 = df_step2.withColumn("loss_close_price",\
                                   when(col("change_price")<0,abs("change_price")).otherwise(0))
    
    #Tính trung bình của cột Gain và cột Loss trong 14 phút gần nhất.
    window_14 = window_spec.rowsBetween(-13,Window.currentRow)
    df_step3 = df_step2.withColumn("avg_gain_14m",\
                                        round(avg("gain_close_price").over(window_14),3))
    df_step3 = df_step3.withColumn("avg_loss_14m",\
                                        round(avg("loss_close_price").over(window_14),3))  
     
    # # Tinh RS 
    df_step4 = df_step3.withColumn("RSI",\
                                   when(col("avg_loss_14m") == 0,100)
                                    .otherwise(round(100-(100/(1+(col("avg_gain_14m") / col("avg_loss_14m")))),3)))                       
    RSI_df = df_step4.select("start_time","symbol", "RSI")
    return RSI_df

In [ ]:
#Draft
from pyspark.sql.functions import row_number,col
from pyspark.sql.window import Window 
window_spec = Window.partitionBy("symbol").orderBy("start_time")
df_test = flatten_df.withColumn("num ID", row_number().over(window_spec))
df_test.filter(col("symbol")=="ETHUSDT").show(truncate=False)

+-------+-------------------+----------+----------+---------+-----------+--------+------+
|symbol |start_time         |open_price|high_price|low_price|close_price|volume  |num ID|
+-------+-------------------+----------+----------+---------+-----------+--------+------+
|ETHUSDT|2026-03-31 18:10:00|2023.26   |2025.91   |2023.26  |2025.65    |168.8016|1     |
|ETHUSDT|2026-03-31 18:11:00|2025.65   |2025.91   |2025.64  |2025.91    |8.585   |2     |
|ETHUSDT|2026-04-03 10:23:00|2057.01   |2057.06   |2057.01  |2057.06    |0.4218  |3     |
|ETHUSDT|2026-04-03 10:23:00|2057.01   |2057.08   |2057.01  |2057.08    |0.7202  |4     |
|ETHUSDT|2026-04-03 10:23:00|2057.01   |2057.09   |2057.01  |2057.09    |0.8531  |5     |
|ETHUSDT|2026-04-03 10:23:00|2057.01   |2057.09   |2056.38  |2056.38    |20.5994 |6     |
|ETHUSDT|2026-04-03 10:23:00|2057.01   |2057.09   |2056.38  |2056.39    |20.6951 |7     |
|ETHUSDT|2026-04-03 10:23:00|2057.01   |2057.09   |2056.38  |2056.39    |24.8317 |8     |
|ETHUSDT|2

In [ ]:
def caculate_MA(batch_df):
    #standard blueprint
    window_spec = Window.partitionBy("symbol").orderBy("start_time")

    #specific blueprint
    window_5m = window_spec.rowsBetween(-4, Window.currentRow)
    window_50m = window_spec.rowsBetween(-49, Window.currentRow)
    window_200m = window_spec.rowsBetween(-199, Window.currentRow)

    #caculate
    ma_df = batch_df.withColumn("MA5", round(avg("close_price").over(window_5m),3))\
            .withColumn("MA50", round(avg("close_price").over(window_50m),3))\
            .withColumn("MA200", round(avg("close_price").over(window_200m),3))
    ma_df = ma_df.select("start_time","symbol", "MA5","MA50","MA200")
    return ma_df

: 

In [ ]:
#Thao
from pyspark.sql.functions import expr, col, avg, when, stddev, lag, sum 
from pyspark.sql.window import Window

def caculate_general(batch_df):
    # cau hinh cua so thoi gian (blueprint)
    window_12 = Window.partitionBy("symbol").orderBy("start_time").rowsBetween(-11, 0)
    window_26 = Window.partitionBy("symbol").orderBy("start_time").rowsBetween(-25, 0)
    window_signal = Window.partitionBy("symbol").orderBy("start_time").rowsBetween(-8, 0)
    window_vol = Window.partitionBy("symbol").orderBy("start_time").rowsBetween(-20, -1)

    # Cấu hình cửa sổ cho Bollinger Bands (Mặc định tính trên chu kỳ SMA 20 phiên)
    window_bb = Window.partitionBy("symbol").orderBy("start_time").rowsBetween(-19, 0)

    # Cấu hình cửa sổ lũy kế vô hạn cho OBV (Quét từ dòng đầu tiên đến dòng hiện tại)
    window_obv = Window.partitionBy("symbol").orderBy("start_time").rowsBetween(Window.unboundedPreceding, 0)

    # Cấu hình cửa sổ lùi 1 dòng duy nhất để so sánh giá phiên hiện tại với phiên trước (Phục vụ OBV)
    window_lag1 = Window.partitionBy("symbol").orderBy("start_time")

    # TIẾN HÀNH TÍNH TOÁN CÁC CHỈ SỐ KỸ THUẬT
    # MACD
    df_indicators = batch_df \
        .withColumn("SMA_12", avg("close_price").over(window_12)) \
        .withColumn("SMA_26", avg("close_price").over(window_26))

    df_indicators = df_indicators.withColumn("MACD_Line", col("SMA_12") - col("SMA_26"))
    df_indicators = df_indicators.withColumn("Signal_Line", avg("MACD_Line").over(window_signal))
    df_indicators = df_indicators.withColumn("MACD_Histogram", round(col("MACD_Line") - col("Signal_Line"),3))

    # VOLUME SPIKE
    df_indicators = df_indicators.withColumn("Avg_Volume_Past", avg("volume").over(window_vol))

    df_indicators = df_indicators.withColumn(
        "Volume_Spike",
        when((col("volume") > (col("Avg_Volume_Past") * 1.5)) & (col("Avg_Volume_Past").isNotNull()), 1).otherwise(0)
    )

    # BOLLINGER BANDS (BB)
    df_indicators = df_indicators.withColumn("BB_Middle", avg("close_price").over(window_bb))
    df_indicators = df_indicators.withColumn("BB_StdDev", stddev("close_price").over(window_bb)) #Standard Deviation
    df_indicators = df_indicators.withColumn("BB_Upper", round(col("BB_Middle") + (col("BB_StdDev") * 2),3))
    df_indicators = df_indicators.withColumn("BB_Lower", round(col("BB_Middle") - (col("BB_StdDev") * 2),3))

    #ON-BALANCE VOLUME (OBV)
    df_indicators = df_indicators.withColumn("prev_close", lag("close_price", 1).over(window_lag1))
    df_indicators = df_indicators.withColumn("direction_volume",
        when(col("close_price") > col("prev_close"), col("volume"))
        .when(col("close_price") < col("prev_close"), -col("volume"))
        .otherwise(0.0)
    )
    df_final = df_indicators.withColumn("OBV", round(sum("direction_volume").over(window_obv),3))

    #KẾT QUẢ:
    df_final = df_final.select(
        "start_time",
        "symbol",
        "MACD_Histogram",
        "Volume_Spike",
        "BB_Upper",
        "BB_Lower",
        "OBV"
    )
    return df_final


In [ ]:
#driven decision
def driven_decision(batch_df):
    #blueprint
    window_spec = Window.partitionBy("symbol").orderBy("start_time")
    general_df = caculate_general(batch_df)
    ma_df = caculate_MA(batch_df)
    rsi_df = caculate_RSI(batch_df)
    # Cấu trúc: df1.join(df2, "tên_cột_khóa\condition", "loại_join")
    caculated_df = batch_df.join(ma_df,["start_time","symbol"],"inner")\
                            .join(general_df,["start_time","symbol"],"inner")\
                            .join(rsi_df,["start_time","symbol"],"inner")
    caculated_df = caculated_df.withColumn("OBV_prv", lag(col("OBV"),1).over(window_spec))
    #trend based on decided algorithm 
    advice_df = caculated_df.withColumn("Advice", 
                                      when((col("close_price") < col("MA50")) | (col("MA50") < col("MA200")),
                                           "❌ DROP ROW. Thị trường không có xu hướng tăng vĩ mô.")
                                      .when((col("close_price") > col("BB_Lower")) | (col("RSI") > 35),
                                            "⏳ KEEP OBSERVING. Giá chưa chiết khấu đủ sâu.Chờ transaction tiếp theo.")
                                      .when((col("MACD_Histogram") <= 0) | (col("close_price") < col("MA5")),
                                            "⏳ WAIT FOR TRIGGER. Lực mua chưa hồi phục. Tiếp tục chờ.")
                                      .when((col("Volume_Spike") == 0) & (col("OBV") <= col("OBV_prv")), 
                                            "⚠️ LOW CONFIDENCE LONG.\nTín hiệu kỹ thuật đẹp nhưng thiếu dòng tiền lớn bảo kê.\n-> Hành động: Bỏ qua lệnh HOẶC chỉ đi tiền 10% volume mục tiêu.")
                                      .otherwise("🚀 HIGH CONFIDENCE LONG.\nCá mập xác nhận đổ tiền đẩy giá (Khối lượng đột biến hoặc OBV phá đỉnh).")
                                      )
      
    return advice_df.drop("OBV_prv")

driven_decision(watermark_df).show(truncate = False)

NameError: name 'watermark_df' is not defined